<a href="https://colab.research.google.com/github/StrawEater/PracticasPDI3erBimestre/blob/main/TP_Rompecabezas_Colab_V2_ejercicios_secciones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

***Nombre de Grupo:***


***Integrantes:***


# TP Rompecabezas Final | Ejercicios por Sección

En este Colab tendras 10 secciones independientes, cada una cuenta con su propia combinacion de degradadores en los que se basara sus rompecabezas. En cada seccion tendran la oportunidad para definir una funcino de preprocesamiento de cada pieza y una funcion de comparacion para calcular que tan distintas son.
Cada seccion cuenta con una validacion para testear sus implementaciones, pasando rompecabezas construidos con distinta semilla y imagenes de base. Por cada una de los rompecabezas se calculara el puntaje promedio de *Vecindad*.
Si el puntaje de vecindad es >= 85% entonces la seccion esta aprobada/superada. Para aprobar el tp necesitaran al menos 6 secciones resueltas.

Las secciones se pueden resolver en cualquier orden.

**Funciones a implementar en cada seccion**:
- `preprocesar_<Seccion>(pieza)` : limpia / normaliza / rota **una** `Pieza` (in place).
- `comparar_<Seccion>(pieza_a, pieza_b, relacion) -> float` : costo de poner `pieza_b` al
  lado de `pieza_a` (menor = más compatibles).

In [1]:
import os, sys
from pathlib import Path

if 'google.colab' in str(get_ipython()):
    if not os.path.exists('core'):
        !git clone https://github.com/MarioSigal/TP_Rompecabezas.git repo_tp
        %cd repo_tp
        !git checkout main
    sys.path.insert(0, os.getcwd())
else:
    raiz = Path.cwd()
    if (raiz / 'core').exists():
        sys.path.insert(0, str(raiz))
    elif (raiz / 'TP_Rompecabezas' / 'core').exists():
        sys.path.insert(0, str(raiz / 'TP_Rompecabezas'))


Cloning into 'repo_tp'...
remote: Enumerating objects: 496, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 496 (delta 78), reused 60 (delta 59), pack-reused 402 (from 3)
Receiving objects: 100% (496/496), 390.80 MiB | 21.83 MiB/s, done.
Resolving deltas: 100% (226/226), done.
Updating files: 100% (192/192), done.
/content/repo_tp
Already on 'main'
Your branch is up to date with 'origin/main'.


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from core.preparacion_imagenes import preparar_imagen_base
from core.RompecabezasV2 import RompecabezasV2, Pieza
from core.reconstructor import reconstruir_desde_afinidades
from core.metricas import calcular_precision_vecindad
from core.detector_forma import pasar_borde_a_1d
from core.bordes import BORDES_ENFRENTADOS, compatibilidad_baseline_por_pieza, construir_matrices_afinidad_desde_piezas
from utils import mostrar_piezas_desordenadas_v2
from core.degradacionesV2 import reiniciar_estado_degradador_por_pieza
from core.degradadores_secciones import (
    degradador_seccion_1,
    degradadorPorXoR,
    degradadorLuminancia,
    degradadorPorCrominancia,
    degradador_seccion_3,
    degradador_seccion_5,
    degradador_color_fourier,
    degradador_color_rotacion,
    degradador_global_sin_impulsivo,
    DegradadorPorPiezaCompuesto
)

GRID = 6
TAMANO_OBJETIVO = (768, 768)
EXTENSIONES_VALIDAS = (".png", ".jpg", ".jpeg")

def listar_imagenes(raiz: str = "imagenes") -> list:
    """Todas las imagenes bajo `raiz`, recursivamente (incluye subcarpetas)."""
    return sorted(str(p) for p in Path(raiz).rglob("*") if p.suffix.lower() in EXTENSIONES_VALIDAS)

def cargar_puzzle(ruta: str, semilla: int, tiene_ranuras: bool, degradador_global=None, degradador_por_pieza=None) -> RompecabezasV2:
    """Reinicia el estado del degradador por pieza (evita reusar la asignacion/argumentos de
    la imagen anterior, ya que los degradadores de degradadores_secciones.py son instancias
    unicas a nivel de modulo) y arma un RompecabezasV2 de GRIDxGRID a partir de la imagen."""
    reiniciar_estado_degradador_por_pieza(degradador_por_pieza)
    imagen = preparar_imagen_base(ruta, tamaño_objetivo=TAMANO_OBJETIVO)
    return RompecabezasV2(
        imagen, cantidad_filas=GRID, cantidad_columnas=GRID, tiene_ranuras=tiene_ranuras,
        degradador_global=degradador_global, degradador_por_pieza=degradador_por_pieza, semilla=semilla,
    )

def perfil_lado(pieza, lado: str) -> dict:
    """Perfil 1D + tipo (PLANO/SALIENTE/ENTRANTE) de un lado, desde la geometria ya conocida
    de la pieza (pieza.geometria.borde_*) — no hace falta re-detectar nada por pixeles."""
    puntos = getattr(pieza.geometria, f"borde_{lado.lower()}")
    centro_pieza = np.mean(pieza.geometria.puntos_esquinas, axis=0)
    return pasar_borde_a_1d(puntos, lado, centro_referencia=centro_pieza)

COLOR_BORDE = {"NORTE": "red", "SUR": "blue", "ESTE": "green", "OESTE": "orange"}

def _fondo_a_cuadros(alto: int, ancho: int, tamano_cuadro: int = 8) -> np.ndarray:
    """Patron de cuadros grises, como el fondo "transparente" de un editor de imagenes."""
    fila = (np.arange(alto)[:, None] // tamano_cuadro) % 2
    columna = (np.arange(ancho)[None, :] // tamano_cuadro) % 2
    es_cuadro_claro = (fila + columna) % 2
    return np.where(es_cuadro_claro[..., None], 0.9, 0.8) * np.ones((alto, ancho, 3))

def mostrar_pieza(pieza, titulo: str = "Pieza", ax=None):
    """Muestra devolver_pieza() con su transparencia real (canal alfa): fuera de la
    silueta de la pieza alfa=0, no es negro. Se grafica sobre un fondo a cuadros para que
    se note la diferencia entre "transparente" y "pixel negro dentro de la pieza"."""
    if ax is None:
        _, ax = plt.subplots(figsize=(4, 4))

    parche = pieza.devolver_pieza()
    color = np.clip(parche[..., :3], 0.0, 1.0)
    alfa = parche[..., -1:].astype(np.float64)
    alfa = alfa / alfa.max() if alfa.max() > 0 else alfa
    alto, ancho = color.shape[:2]

    ax.imshow(_fondo_a_cuadros(alto, ancho))
    ax.imshow(np.dstack([color, alfa]))

    ax.set_title(titulo, fontsize=10)
    ax.axis("off")
    return ax

def graficar_pieza_con_bordes(pieza, titulo: str, ax=None):
    """Dibuja la pieza (con transparencia real, ver mostrar_pieza) y superpone sus 4
    bordes (pieza.geometria.borde_*): devolver_pieza() vive en el mismo sistema de
    coordenadas que esos bordes, asi que se pueden superponer directamente, sin ninguna
    cuenta de offset."""
    if ax is None:
        _, ax = plt.subplots(figsize=(4, 4))

    mostrar_pieza(pieza, titulo, ax=ax)

    for lado, color in COLOR_BORDE.items():
        puntos = getattr(pieza.geometria, f"borde_{lado.lower()}")
        ax.plot(puntos[:, 0], puntos[:, 1], color=color, linewidth=2.5, label=lado)

    ax.set_title(titulo, fontsize=10)
    ax.legend(loc="upper right", fontsize=7)
    ax.axis("off")
    return ax

def resolver_seccion(rompecabezas: RompecabezasV2, preprocesar_pieza, comparar_piezas) -> np.ndarray:
    """Arma la grilla propuesta (cantidad_filas x cantidad_columnas, con el id de pieza en
    cada celda) usando las dos funciones de la seccion: `preprocesar_pieza(pieza)` se aplica
    una vez a cada pieza (in place) y `comparar_piezas(pieza_a, pieza_b, relacion)` da el
    costo de cada par para construir las matrices de afinidad."""
    for pieza in rompecabezas.piezas:
        preprocesar_pieza(pieza)
    matrices = construir_matrices_afinidad_desde_piezas(rompecabezas.piezas, funcion_compatibilidad=comparar_piezas)
    return reconstruir_desde_afinidades(matrices, rompecabezas.cantidad_filas, rompecabezas.cantidad_columnas)

def reportar_seccion(nombre: str, resultados: list, umbral: float = 0.85) -> float:
    print(f"{'Imagen':<55} | {'Semilla':>7} | {'Vecindad':>9}")
    print("-" * 78)
    for r in resultados:
        print(f"{Path(r['imagen']).name:<55} | {r['semilla']:>7} | {r['vecindad']*100:>8.1f}%")

    promedio = float(np.mean([r["vecindad"] for r in resultados]))
    buenas = [r["imagen"] for r in resultados if r["vecindad"] >= umbral]
    print("-" * 78)
    print(f"Promedio de precision de vecindad: {promedio*100:.1f}%")
    print(f"Imagenes con vecindad >= {umbral*100:.0f}%: {len(buenas)}/{len(resultados)}")
    for imagen in buenas:
        print(f"  - {imagen}")
    return promedio

print('Módulos cargados. Imagenes encontradas:', len(listar_imagenes()))


Módulos cargados. Imagenes encontradas: 162


## La interfaz de `Pieza`

Todo lo que hace falta para resolver cualquier sección vive en `rompecabezas.piezas` (una
tupla de objetos `Pieza`) y en `pieza.geometria`.

**`Pieza`**
- `pieza.id`: índice 0..N-1. Coincide con las celdas de `rompecabezas.obtener_matriz_correcta()`. Es solo un identificador, es trampa usarla para obtener el ordenamiento correcto
- `pieza.imagen`: array RGB `float64` en `[0, 1]`, con el padding (fondo negro) alrededor de la forma real.
- `pieza.angulo`: rotación acumulada en grados (0 si nunca se rotó). Es solo para debuggear, no la utilizen para resolver una seccion.
- `pieza.devolver_pieza() -> np.ndarray`: recorte RGBA ajustado a la forma real (dentro de la mascara definida por `pieza.geometria`).
- `pieza.actualizar_pieza_recortada(parche_color)`: recibe un array del mismo tamaño (sin canal transparente) y lo escribe de vuelta en `pieza.imagen` respetando la máscara.
- `pieza.rotar(angulo_grados)`: rota la imagen Y la geometría de forma consistente.
- `pieza.calcular_borde_color(lado)`: color de `pieza.imagen` a lo largo de uno de los 4 lados (`"NORTE"/"SUR"/"ESTE"/"OESTE"`), en el mismo orden que `pieza.geometria.borde_<lado>`.

**`pieza.geometria` (`MascaraMuesca`)**
- `borde_norte` / `borde_sur` / `borde_este` / `borde_oeste` — arrays Nx2 con los puntos de contorno de cada lado (rectos si `tiene_ranuras=False`, con encastres si `tiene_ranuras=True`).
- `puntos_esquinas` — las 4 esquinas de la pieza (TL, TR, BR, BL).
- `generar_mascara()` — máscara booleana de la silueta.
- `bounding_box_real()` — el rectángulo "sin encastres" de la pieza.

**Funciones útiles**
- `pasar_borde_a_1d(puntos_borde, lado, centro_referencia=centro)` — convierte un `borde_*` en una señal 1D: `{"type": "PLANO"/"SALIENTE"/"ENTRANTE", "profile": ...}`.

In [3]:
# Demo rapida: no hace falta correrla para resolver las secciones, es solo para ver la interfaz.
_pieza_demo = cargar_puzzle(listar_imagenes()[0], semilla=0, tiene_ranuras=True).piezas[0]

print('id:', _pieza_demo.id, '| angulo:', _pieza_demo.angulo)
print('imagen.shape (con padding):', _pieza_demo.imagen.shape, _pieza_demo.imagen.dtype)
print()
_parche_demo = _pieza_demo.devolver_pieza()
print('devolver_pieza().shape (RGBA, forma real):', _parche_demo.shape)
print()
print('borde_este, primeros 3 puntos:\n', _pieza_demo.geometria.borde_este[:3])
print('puntos_esquinas (TL, TR, BR, BL):\n', _pieza_demo.geometria.puntos_esquinas)
print()
_centro_demo = np.mean(_pieza_demo.geometria.puntos_esquinas, axis=0)
_perfil_este_demo = pasar_borde_a_1d(_pieza_demo.geometria.borde_este, "ESTE", centro_referencia=_centro_demo)
print('tipo de lado ESTE:', _perfil_este_demo["type"])
print()
# Cualquier pieza se puede rotar a un angulo arbitrario (no solo 0/90/180/270). rotar()
# actualiza la imagen Y pieza.geometria.borde_*/puntos_esquinas de forma consistente
# (se rotan analiticamente los puntos ya conocidos, no hace falta re-detectar la forma).
print('--- pieza.rotar(angulo) ---')
print('angulo antes:', _pieza_demo.angulo)
_pieza_demo.rotar(30.0)
print('angulo despues de rotar(30.0):', _pieza_demo.angulo)
print('imagen.shape despues de rotar (crece para no recortar esquinas):', _pieza_demo.imagen.shape)
print('borde_este ya rotado, primeros 3 puntos:\n', _pieza_demo.geometria.borde_este[:3])
print()
# Color a lo largo de un lado (para comparar con el lado de la pieza vecina): un array
# Nx3 RGB, un color por punto de pieza.geometria.borde_<lado>, en el mismo orden.
_color_este_demo = _pieza_demo.calcular_borde_color("ESTE")
print('calcular_borde_color("ESTE").shape:', _color_este_demo.shape)
print('primeros 3 colores:\n', _color_este_demo[:3])
print()
del _pieza_demo, _parche_demo, _centro_demo, _perfil_este_demo, _color_este_demo


id: 0 | angulo: 0
imagen.shape (con padding): (213, 219, 3) float64

devolver_pieza().shape (RGBA, forma real): (129, 129, 4)

borde_este, primeros 3 puntos:
 [[128.   0.]
 [128.   1.]
 [128.   2.]]
puntos_esquinas (TL, TR, BR, BL):
 [[  0   0]
 [128   0]
 [128 128]
 [  0 128]]

tipo de lado ESTE: ENTRANTE

--- pieza.rotar(angulo) ---
angulo antes: 0
angulo despues de rotar(30.0): 30.0
imagen.shape despues de rotar (crece para no recortar esquinas): (295, 301, 3)
borde_este ya rotado, primeros 3 puntos:
 [[174.85125168  64.        ]
 [174.35125168  64.8660254 ]
 [173.85125168  65.73205081]]

calcular_borde_color("ESTE").shape: (128, 3)
primeros 3 colores:
 [[0.         0.41960785 0.56078433]
 [0.         0.41960783 0.5607843 ]
 [0.         0.41960784 0.56078431]]



### La pieza tiene transparencia real

`devolver_pieza()`: fuera de la silueta el canal alfa
es 0 (transparente).

In [ ]:
_pieza_alfa = cargar_puzzle(listar_imagenes()[0], semilla=0, tiene_ranuras=True).piezas[14]
mostrar_pieza(_pieza_alfa, "Pieza con transparencia real (canal alfa)")
plt.show()

del _pieza_alfa


### Viendo la pieza y sus bordes

Una pieza interior (fila 2, columna 2 de la grilla 6x6) antes y después de `rotar()`: el círculo marca el primer punto de cada borde. Los bordes quedan pegados al contorno real de la pieza en los dos casos, porque `rotar()` rota la geometría junto con la imagen.

In [ ]:
_rc_plot = cargar_puzzle(listar_imagenes()[0], semilla=0, tiene_ranuras=True)
_pieza_plot = _rc_plot.piezas[14]  # fila 2, columna 2 en una grilla 6x6: pieza interior

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
graficar_pieza_con_bordes(_pieza_plot, "Antes de rotar", ax=axes[0])

_pieza_plot.rotar(35.0)
graficar_pieza_con_bordes(_pieza_plot, "Después de rotar(35°)", ax=axes[1])

plt.tight_layout()
plt.show()

# Un solo borde, como señal 1D (pasar_borde_a_1d): asi se ve una muesca SALIENTE/ENTRANTE.
_perfil_plot = perfil_lado(_pieza_plot, "ESTE")
plt.figure(figsize=(5, 2.5))
plt.plot(_perfil_plot["profile"])
plt.title(f'Perfil del borde ESTE (tipo: {_perfil_plot["type"]})', fontsize=10)
plt.xlabel("posición a lo largo del borde")
plt.ylabel("desviación perpendicular")
plt.tight_layout()
plt.show()

del _rc_plot, _pieza_plot, fig, axes, _perfil_plot


## Ejemplo: una función de comparación

La firma que espera `construir_matrices_afinidad_desde_piezas` es siempre la misma:
`mi_funcion_compatibilidad(pieza_a, pieza_b, relacion) -> costo` (float), donde `relacion` es
`"horizontal"` (B a la derecha de A) o `"vertical"` (B abajo de A), y **menor costo = más
compatibles**. `BORDES_ENFRENTADOS[relacion]` da el par de lados que se tocan en cada caso.

Ejemplo

In [ ]:
def mi_funcion_compatibilidad(pieza_a, pieza_b, relacion):
    lado_a, lado_b = BORDES_ENFRENTADOS[relacion]  # ej: ("ESTE", "OESTE") si relacion == "horizontal"

    banda_a = pieza_a.calcular_borde_color(lado_a)
    banda_b = pieza_b.calcular_borde_color(lado_b)

    # Los dos lados pueden tener distinta cantidad de puntos (long. de borde distinta): se
    # remuestrea al mismo largo antes de comparar.
    n = min(len(banda_a), len(banda_b))
    idx_a = np.linspace(0, len(banda_a) - 1, n).astype(int)
    idx_b = np.linspace(0, len(banda_b) - 1, n).astype(int)

    return float(np.mean((banda_a[idx_a] - banda_b[idx_b]) ** 2))


# Asi se usa una funcion de comparacion: construir_matrices_afinidad_desde_piezas +
# reconstruir_desde_afinidades. En cada seccion de abajo eso lo hace resolver_seccion, con
# tu preprocesar_<Seccion> y tu comparar_<Seccion>.
_rc_demo = cargar_puzzle(listar_imagenes()[0], semilla=0, tiene_ranuras=False)
_matrices_demo = construir_matrices_afinidad_desde_piezas(_rc_demo.piezas, funcion_compatibilidad=mi_funcion_compatibilidad)
print('matriz horizontal shape:', _matrices_demo["horizontal"].shape)
print('costo pieza 0 -> pieza 1 (horizontal):', _matrices_demo["horizontal"][0, 1])

_grilla_demo = reconstruir_desde_afinidades(_matrices_demo, _rc_demo.cantidad_filas, _rc_demo.cantidad_columnas)
print('grilla propuesta:\n', _grilla_demo)

del _rc_demo, _matrices_demo, _grilla_demo


## Sección 1 — GlobalNoise

- Degradación: `degradador_seccion_1` (global, se aplica sobre toda la imagen antes de cortar las piezas). Mezcla de ruido continuo (Gaussiano/Uniforme/Rayleigh) + sal y pimienta.
- Piezas cuadradas (`tiene_ranuras=False`), grilla 6x6.
- Implementar: `preprocesar_GlobalNoise(pieza)` y `comparar_GlobalNoise(pieza_a, pieza_b, relacion) -> float`.
  
  La grilla la arma `resolver_seccion(rc, preprocesar_GlobalNoise, comparar_GlobalNoise)`.

In [ ]:
# Ejemplo de rompecabezas de esta seccion (primera imagen, semilla 100)
rc_ejemplo_GlobalNoise = cargar_puzzle(listar_imagenes("imagenes/seccion_1")[0], 100, tiene_ranuras=False, degradador_global=degradador_seccion_1)
mostrar_piezas_desordenadas_v2(rc_ejemplo_GlobalNoise, max_piezas=12, titulo="Sección 1 — GlobalNoise")


In [ ]:
def preprocesar_GlobalNoise(pieza: Pieza) -> None:
    """Prepara `pieza` IN PLACE (no devuelve nada): limpiar con pieza.actualizar_pieza_recortada(...),
    rotar con pieza.rotar(...), etc. Se llama una vez por pieza, antes de comparar."""
    ###COMPLETAR
    raise NotImplementedError

def comparar_GlobalNoise(pieza_a: Pieza, pieza_b: Pieza, relacion: str) -> float:
    """Costo de poner pieza_b junto a pieza_a (menor = mas compatibles). `relacion` es
    "horizontal" (B a la derecha de A) o "vertical" (B abajo de A); BORDES_ENFRENTADOS[relacion]
    da el par de lados que se tocan. Las piezas ya pasaron por preprocesar_GlobalNoise."""
    ###COMPLETAR
    raise NotImplementedError


In [ ]:
resultados_GlobalNoise = []
for i, ruta in enumerate(listar_imagenes("imagenes/seccion_1")):
    semilla = 100 + i
    rc = cargar_puzzle(ruta, semilla, tiene_ranuras=False, degradador_global=degradador_seccion_1)
    grilla = resolver_seccion(rc, preprocesar_GlobalNoise, comparar_GlobalNoise)
    resultados_GlobalNoise.append({"imagen": ruta, "semilla": semilla, "vecindad": calcular_precision_vecindad(grilla, rc)})


In [ ]:
promedio_GlobalNoise = reportar_seccion("GlobalNoise", resultados_GlobalNoise)
assert promedio_GlobalNoise > 0.85, (
    f"Seccion GlobalNoise: el promedio de precision de vecindad ({promedio_GlobalNoise*100:.1f}%) "
    "debe superar el 85%."
)
print("Seccion GlobalNoise: OK")


## Sección 2 — ColorDegradation

- Degradación: `degradadorPorXoR/Luminanca/Crominancia` (por pieza).
- Piezas cuadradas (`tiene_ranuras=False`), grilla 6x6.
- Implementar: `preprocesar_ColorDegradation(pieza)` y `comparar_ColorDegradation(pieza_a, pieza_b, relacion) -> float`.

In [ ]:
# Ejemplo de rompecabezas de esta seccion (primera imagen, semilla 200)
rc_ejemplo_ColorDegradation = cargar_puzzle(listar_imagenes("imagenes/seccion_2")[0], 200, tiene_ranuras=False, degradador_por_pieza=degradadorPorXoR)
mostrar_piezas_desordenadas_v2(rc_ejemplo_ColorDegradation, max_piezas=12, titulo="Sección 2 — ColorDegradation")


In [ ]:
def preprocesar_ColorDegradation(pieza: Pieza) -> None:
    """Prepara `pieza` IN PLACE (no devuelve nada): limpiar con pieza.actualizar_pieza_recortada(...),
    rotar con pieza.rotar(...), etc. Se llama una vez por pieza, antes de comparar."""
    ###COMPLETAR
    raise NotImplementedError

def comparar_ColorDegradation(pieza_a: Pieza, pieza_b: Pieza, relacion: str) -> float:
    """Costo de poner pieza_b junto a pieza_a (menor = mas compatibles). `relacion` es
    "horizontal" (B a la derecha de A) o "vertical" (B abajo de A); BORDES_ENFRENTADOS[relacion]
    da el par de lados que se tocan. Las piezas ya pasaron por preprocesar_ColorDegradation."""
    ###COMPLETAR
    raise NotImplementedError


In [ ]:
resultados_ColorDegradation = []
for i, ruta in enumerate(listar_imagenes("imagenes/seccion_2")):
    semilla = 200 + i

    rng = np.random.default_rng(seed=semilla)
    degradadores_color = [degradadorPorXoR, degradadorLuminancia, degradadorPorCrominancia]
    degradador = rng.choice(degradadores_color)

    rc = cargar_puzzle(ruta, semilla, tiene_ranuras=False, degradador_por_pieza=degradador)
    grilla = resolver_seccion(rc, preprocesar_ColorDegradation, comparar_ColorDegradation)
    resultados_ColorDegradation.append({"imagen": ruta, "semilla": semilla, "vecindad": calcular_precision_vecindad(grilla, rc)})


In [ ]:
promedio_ColorDegradation = reportar_seccion("ColorDegradation", resultados_ColorDegradation)
assert promedio_ColorDegradation > 0.85, (
    f"Seccion ColorDegradation: el promedio de precision de vecindad ({promedio_ColorDegradation*100:.1f}%) "
    "debe superar el 85%."
)
print("Seccion ColorDegradation: OK")


## Sección 3 — Fourier

- Degradación: `degradador_seccion_3` (por pieza), `DegradadorAleatorio` entre 5 tramas
  periódicas Fourier (cuádruple/diagonal/doble frecuencia/oblicua/ortogonal).
- Piezas cuadradas (`tiene_ranuras=False`), grilla 6x6.
- Implementar: `preprocesar_Fourier(pieza)` y `comparar_Fourier(pieza_a, pieza_b, relacion) -> float`.

In [ ]:
# Ejemplo de rompecabezas de esta seccion (primera imagen, semilla 300)
rc_ejemplo_Fourier = cargar_puzzle(listar_imagenes("imagenes/seccion_3")[0], 300, tiene_ranuras=False, degradador_por_pieza=degradador_seccion_3)
mostrar_piezas_desordenadas_v2(rc_ejemplo_Fourier, max_piezas=12, titulo="Sección 3 — Fourier")


In [ ]:
def preprocesar_Fourier(pieza: Pieza) -> None:
    """Prepara `pieza` IN PLACE (no devuelve nada): limpiar con pieza.actualizar_pieza_recortada(...),
    rotar con pieza.rotar(...), etc. Se llama una vez por pieza, antes de comparar."""
    ###COMPLETAR
    raise NotImplementedError

def comparar_Fourier(pieza_a: Pieza, pieza_b: Pieza, relacion: str) -> float:
    """Costo de poner pieza_b junto a pieza_a (menor = mas compatibles). `relacion` es
    "horizontal" (B a la derecha de A) o "vertical" (B abajo de A); BORDES_ENFRENTADOS[relacion]
    da el par de lados que se tocan. Las piezas ya pasaron por preprocesar_Fourier."""
    ###COMPLETAR
    raise NotImplementedError


In [ ]:
resultados_Fourier = []
for i, ruta in enumerate(listar_imagenes("imagenes/seccion_3")):
    semilla = 300 + i
    rc = cargar_puzzle(ruta, semilla, tiene_ranuras=False, degradador_por_pieza=degradador_seccion_3)
    grilla = resolver_seccion(rc, preprocesar_Fourier, comparar_Fourier)
    resultados_Fourier.append({"imagen": ruta, "semilla": semilla, "vecindad": calcular_precision_vecindad(grilla, rc)})


In [ ]:
promedio_Fourier = reportar_seccion("Fourier", resultados_Fourier)
assert promedio_Fourier > 0.85, (
    f"Seccion Fourier: el promedio de precision de vecindad ({promedio_Fourier*100:.1f}%) "
    "debe superar el 85%."
)
print("Seccion Fourier: OK")


## Sección 4 — Geometry

- Sin degradación de señal: piezas con encastres reales (`tiene_ranuras=True`), sin ruido,
  color ni rotación.
- Grilla 6x6.
- Implementar: `preprocesar_Geometry(pieza)` y `comparar_Geometry(pieza_a, pieza_b, relacion) -> float`.

In [ ]:
# Ejemplo de rompecabezas de esta seccion (primera imagen, semilla 400)
rc_ejemplo_Geometry = cargar_puzzle(listar_imagenes("imagenes/seccion_4")[0], 400, tiene_ranuras=True)
mostrar_piezas_desordenadas_v2(rc_ejemplo_Geometry, max_piezas=12, titulo="Sección 4 — Geometry")


In [ ]:
def preprocesar_Geometry(pieza: Pieza) -> None:
    """Prepara `pieza` IN PLACE (no devuelve nada): limpiar con pieza.actualizar_pieza_recortada(...),
    rotar con pieza.rotar(...), etc. Se llama una vez por pieza, antes de comparar."""
    ###COMPLETAR
    raise NotImplementedError

def comparar_Geometry(pieza_a: Pieza, pieza_b: Pieza, relacion: str) -> float:
    """Costo de poner pieza_b junto a pieza_a (menor = mas compatibles). `relacion` es
    "horizontal" (B a la derecha de A) o "vertical" (B abajo de A); BORDES_ENFRENTADOS[relacion]
    da el par de lados que se tocan. Las piezas ya pasaron por preprocesar_Geometry."""
    ###COMPLETAR
    raise NotImplementedError


In [ ]:
resultados_Geometry = []
for i, ruta in enumerate(listar_imagenes("imagenes/seccion_4")):
    semilla = 400 + i
    rc = cargar_puzzle(ruta, semilla, tiene_ranuras=True)
    grilla = resolver_seccion(rc, preprocesar_Geometry, comparar_Geometry)
    resultados_Geometry.append({"imagen": ruta, "semilla": semilla, "vecindad": calcular_precision_vecindad(grilla, rc)})


In [ ]:
promedio_Geometry = reportar_seccion("Geometry", resultados_Geometry)
assert promedio_Geometry > 0.85, (
    f"Seccion Geometry: el promedio de precision de vecindad ({promedio_Geometry*100:.1f}%) "
    "debe superar el 85%."
)
print("Seccion Geometry: OK")


## Sección 5 — Rotation

- Degradación: `degradador_seccion_5` (por pieza), `DegradadorRotacionYLineasAlMismoAngulo`:
  rota cada pieza un ángulo aleatorio y agrega líneas de referencia al mismo ángulo.
- Piezas con encastres reales (`tiene_ranuras=True`), grilla 6x6.
- Implementar: `preprocesar_Rotation(pieza)` y `comparar_Rotation(pieza_a, pieza_b, relacion) -> float`.

In [ ]:
# Ejemplo de rompecabezas de esta seccion (primera imagen, semilla 500)
rc_ejemplo_Rotation = cargar_puzzle(listar_imagenes("imagenes/seccion_5")[0], 500, tiene_ranuras=True, degradador_por_pieza=degradador_seccion_5)
mostrar_piezas_desordenadas_v2(rc_ejemplo_Rotation, max_piezas=12, titulo="Sección 5 — Rotation")


In [ ]:
def preprocesar_Rotation(pieza: Pieza) -> None:
    """Prepara `pieza` IN PLACE (no devuelve nada): limpiar con pieza.actualizar_pieza_recortada(...),
    rotar con pieza.rotar(...), etc. Se llama una vez por pieza, antes de comparar."""
    ###COMPLETAR
    raise NotImplementedError

def comparar_Rotation(pieza_a: Pieza, pieza_b: Pieza, relacion: str) -> float:
    """Costo de poner pieza_b junto a pieza_a (menor = mas compatibles). `relacion` es
    "horizontal" (B a la derecha de A) o "vertical" (B abajo de A); BORDES_ENFRENTADOS[relacion]
    da el par de lados que se tocan. Las piezas ya pasaron por preprocesar_Rotation."""
    ###COMPLETAR
    raise NotImplementedError


In [ ]:
resultados_Rotation = []
for i, ruta in enumerate(listar_imagenes("imagenes/seccion_5")):
    semilla = 500 + i
    rc = cargar_puzzle(ruta, semilla, tiene_ranuras=True, degradador_por_pieza=degradador_seccion_5)
    grilla = resolver_seccion(rc, preprocesar_Rotation, comparar_Rotation)
    resultados_Rotation.append({"imagen": ruta, "semilla": semilla, "vecindad": calcular_precision_vecindad(grilla, rc)})


In [ ]:
promedio_Rotation = reportar_seccion("Rotation", resultados_Rotation)
assert promedio_Rotation > 0.85, (
    f"Seccion Rotation: el promedio de precision de vecindad ({promedio_Rotation*100:.1f}%) "
    "debe superar el 85%."
)
print("Seccion Rotation: OK")


## Sección 6 — GlobalNoise + ColorDegradation

- Degradaciones: `degradador_seccion_1` (global) + `degradadorPorXoR` (por pieza).
- Piezas cuadradas (`tiene_ranuras=False`), grilla 6x6.
- Implementar: `preprocesar_GlobalNoise_ColorDegradation(pieza)` y `comparar_GlobalNoise_ColorDegradation(pieza_a, pieza_b, relacion) -> float`.

In [ ]:
# Ejemplo de rompecabezas de esta seccion (primera imagen, semilla 600)
rc_ejemplo_GlobalNoise_ColorDegradation = cargar_puzzle(listar_imagenes("imagenes/seccion_6")[0], 600, tiene_ranuras=False, degradador_global=degradador_seccion_1, degradador_por_pieza=degradadorPorXoR)
mostrar_piezas_desordenadas_v2(rc_ejemplo_GlobalNoise_ColorDegradation, max_piezas=12, titulo="Sección 6 — GlobalNoise + ColorDegradation")


In [ ]:
def preprocesar_GlobalNoise_ColorDegradation(pieza: Pieza) -> None:
    """Prepara `pieza` IN PLACE (no devuelve nada): limpiar con pieza.actualizar_pieza_recortada(...),
    rotar con pieza.rotar(...), etc. Se llama una vez por pieza, antes de comparar."""
    ###COMPLETAR
    raise NotImplementedError

def comparar_GlobalNoise_ColorDegradation(pieza_a: Pieza, pieza_b: Pieza, relacion: str) -> float:
    """Costo de poner pieza_b junto a pieza_a (menor = mas compatibles). `relacion` es
    "horizontal" (B a la derecha de A) o "vertical" (B abajo de A); BORDES_ENFRENTADOS[relacion]
    da el par de lados que se tocan. Las piezas ya pasaron por preprocesar_GlobalNoise_ColorDegradation."""
    ###COMPLETAR
    raise NotImplementedError


In [ ]:
resultados_GlobalNoise_ColorDegradation = []
for i, ruta in enumerate(listar_imagenes("imagenes/seccion_6")):
    semilla = 600 + i

    rng = np.random.default_rng(seed=semilla)
    degradadores_color = [degradadorPorXoR, degradadorLuminancia, degradadorPorCrominancia]
    degradador = rng.choice(degradadores_color)

    rc = cargar_puzzle(ruta, semilla, tiene_ranuras=False, degradador_global=degradador_seccion_1, degradador_por_pieza=degradador)
    grilla = resolver_seccion(rc, preprocesar_GlobalNoise_ColorDegradation, comparar_GlobalNoise_ColorDegradation)
    resultados_GlobalNoise_ColorDegradation.append({"imagen": ruta, "semilla": semilla, "vecindad": calcular_precision_vecindad(grilla, rc)})


In [ ]:
promedio_GlobalNoise_ColorDegradation = reportar_seccion("GlobalNoise_ColorDegradation", resultados_GlobalNoise_ColorDegradation)
assert promedio_GlobalNoise_ColorDegradation > 0.85, (
    f"Seccion GlobalNoise_ColorDegradation: el promedio de precision de vecindad ({promedio_GlobalNoise_ColorDegradation*100:.1f}%) "
    "debe superar el 85%."
)
print("Seccion GlobalNoise_ColorDegradation: OK")


## Sección 7 — ColorDegradation + Fourier

- Degradación: `degradador_color_fourier` (por pieza): `DegradadorPorPiezaCompuesto`, aplica
  `degradadorPorXoR` y después `degradador_seccion_3`, en ese orden, sobre la misma pieza.
- Piezas cuadradas (`tiene_ranuras=False`), grilla 6x6.
- Implementar: `preprocesar_ColorDegradation_Fourier(pieza)` y `comparar_ColorDegradation_Fourier(pieza_a, pieza_b, relacion) -> float`.

In [ ]:
# Ejemplo de rompecabezas de esta seccion (primera imagen, semilla 700)
rc_ejemplo_ColorDegradation_Fourier = cargar_puzzle(listar_imagenes("imagenes/seccion_7")[0], 700, tiene_ranuras=False, degradador_por_pieza=degradador_color_fourier)
mostrar_piezas_desordenadas_v2(rc_ejemplo_ColorDegradation_Fourier, max_piezas=12, titulo="Sección 7 — ColorDegradation + Fourier")


In [ ]:
def preprocesar_ColorDegradation_Fourier(pieza: Pieza) -> None:
    """Prepara `pieza` IN PLACE (no devuelve nada): limpiar con pieza.actualizar_pieza_recortada(...),
    rotar con pieza.rotar(...), etc. Se llama una vez por pieza, antes de comparar."""
    ###COMPLETAR
    raise NotImplementedError

def comparar_ColorDegradation_Fourier(pieza_a: Pieza, pieza_b: Pieza, relacion: str) -> float:
    """Costo de poner pieza_b junto a pieza_a (menor = mas compatibles). `relacion` es
    "horizontal" (B a la derecha de A) o "vertical" (B abajo de A); BORDES_ENFRENTADOS[relacion]
    da el par de lados que se tocan. Las piezas ya pasaron por preprocesar_ColorDegradation_Fourier."""
    ###COMPLETAR
    raise NotImplementedError


In [ ]:
resultados_ColorDegradation_Fourier = []
for i, ruta in enumerate(listar_imagenes("imagenes/seccion_7")):
    semilla = 700 + i

    rng = np.random.default_rng(seed=semilla)
    degradadores_color = [degradadorPorXoR, degradadorLuminancia, degradadorPorCrominancia]
    degradador = rng.choice(degradadores_color)

    degradador_seccion_7 = DegradadorPorPiezaCompuesto(degradador, degradador_seccion_3)

    rc = cargar_puzzle(ruta, semilla, tiene_ranuras=False, degradador_por_pieza=degradador_seccion_7)
    grilla = resolver_seccion(rc, preprocesar_ColorDegradation_Fourier, comparar_ColorDegradation_Fourier)
    resultados_ColorDegradation_Fourier.append({"imagen": ruta, "semilla": semilla, "vecindad": calcular_precision_vecindad(grilla, rc)})


In [ ]:
promedio_ColorDegradation_Fourier = reportar_seccion("ColorDegradation_Fourier", resultados_ColorDegradation_Fourier)
assert promedio_ColorDegradation_Fourier > 0.85, (
    f"Seccion ColorDegradation_Fourier: el promedio de precision de vecindad ({promedio_ColorDegradation_Fourier*100:.1f}%) "
    "debe superar el 85%."
)
print("Seccion ColorDegradation_Fourier: OK")


## Sección 8 — GlobalNoise + ColorDegradation + Fourier (sin sal y pimienta)

- Degradaciones, en este orden: `degradador_global_sin_impulsivo` (global, ruido continuo
  SIN sal y pimienta) → `degradador_color_fourier` (por pieza: color y después Fourier).
- Piezas cuadradas (`tiene_ranuras=False`), grilla 6x6.
- Implementar: `preprocesar_GlobalNoise_ColorDegradation_Fourier(pieza)` y `comparar_GlobalNoise_ColorDegradation_Fourier(pieza_a, pieza_b, relacion) -> float`.
  La grilla la arma `resolver_seccion(rc, preprocesar_GlobalNoise_ColorDegradation_Fourier, comparar_GlobalNoise_ColorDegradation_Fourier)`.

In [ ]:
# Ejemplo de rompecabezas de esta seccion (primera imagen, semilla 800)
rc_ejemplo_GlobalNoise_ColorDegradation_Fourier = cargar_puzzle(listar_imagenes("imagenes/seccion_8")[0], 800, tiene_ranuras=False, degradador_global=degradador_global_sin_impulsivo, degradador_por_pieza=degradador_color_fourier)
mostrar_piezas_desordenadas_v2(rc_ejemplo_GlobalNoise_ColorDegradation_Fourier, max_piezas=12, titulo="Sección 8 — GlobalNoise + ColorDegradation + Fourier (sin sal y pimienta)")


In [ ]:
def preprocesar_GlobalNoise_ColorDegradation_Fourier(pieza: Pieza) -> None:
    """Prepara `pieza` IN PLACE (no devuelve nada): limpiar con pieza.actualizar_pieza_recortada(...),
    rotar con pieza.rotar(...), etc. Se llama una vez por pieza, antes de comparar."""
    ###COMPLETAR
    raise NotImplementedError

def comparar_GlobalNoise_ColorDegradation_Fourier(pieza_a: Pieza, pieza_b: Pieza, relacion: str) -> float:
    """Costo de poner pieza_b junto a pieza_a (menor = mas compatibles). `relacion` es
    "horizontal" (B a la derecha de A) o "vertical" (B abajo de A); BORDES_ENFRENTADOS[relacion]
    da el par de lados que se tocan. Las piezas ya pasaron por preprocesar_GlobalNoise_ColorDegradation_Fourier."""
    ###COMPLETAR
    raise NotImplementedError


In [ ]:
resultados_GlobalNoise_ColorDegradation_Fourier = []
for i, ruta in enumerate(listar_imagenes("imagenes/seccion_8")):
    semilla = 800 + i

    rng = np.random.default_rng(seed=semilla)
    degradadores_color = [degradadorPorXoR, degradadorLuminancia, degradadorPorCrominancia]
    degradador = rng.choice(degradadores_color)

    degradador_seccion_8 = DegradadorPorPiezaCompuesto(degradador, degradador_seccion_3)

    rc = cargar_puzzle(ruta, semilla, tiene_ranuras=False, degradador_global=degradador_global_sin_impulsivo, degradador_por_pieza=degradador_seccion_8)
    grilla = resolver_seccion(rc, preprocesar_GlobalNoise_ColorDegradation_Fourier, comparar_GlobalNoise_ColorDegradation_Fourier)
    resultados_GlobalNoise_ColorDegradation_Fourier.append({"imagen": ruta, "semilla": semilla, "vecindad": calcular_precision_vecindad(grilla, rc)})


In [ ]:
promedio_GlobalNoise_ColorDegradation_Fourier = reportar_seccion("GlobalNoise_ColorDegradation_Fourier", resultados_GlobalNoise_ColorDegradation_Fourier)
assert promedio_GlobalNoise_ColorDegradation_Fourier > 0.85, (
    f"Seccion GlobalNoise_ColorDegradation_Fourier: el promedio de precision de vecindad ({promedio_GlobalNoise_ColorDegradation_Fourier*100:.1f}%) "
    "debe superar el 85%."
)
print("Seccion GlobalNoise_ColorDegradation_Fourier: OK")


## Sección 9 — GlobalNoise + ColorDegradation + Fourier + Geometry (sin sal y pimienta)

- Degradaciones, en este orden: `degradador_global_sin_impulsivo` (global) →
  `degradador_color_fourier` (por pieza: color y después Fourier).
- Piezas con encastres reales (`tiene_ranuras=True`), grilla 6x6.
- Implementar: `preprocesar_GlobalNoise_ColorDegradation_Fourier_Geometry(pieza)` y `comparar_GlobalNoise_ColorDegradation_Fourier_Geometry(pieza_a, pieza_b, relacion) -> float`.

In [ ]:
# Ejemplo de rompecabezas de esta seccion (primera imagen, semilla 900)
rc_ejemplo_GlobalNoise_ColorDegradation_Fourier_Geometry = cargar_puzzle(listar_imagenes("imagenes/seccion_9")[0], 900, tiene_ranuras=True, degradador_global=degradador_global_sin_impulsivo, degradador_por_pieza=degradador_color_fourier)
mostrar_piezas_desordenadas_v2(rc_ejemplo_GlobalNoise_ColorDegradation_Fourier_Geometry, max_piezas=12, titulo="Sección 9 — GlobalNoise + ColorDegradation + Fourier + Geometry (sin sal y pimienta)")


In [ ]:
def preprocesar_GlobalNoise_ColorDegradation_Fourier_Geometry(pieza: Pieza) -> None:
    """Prepara `pieza` IN PLACE (no devuelve nada): limpiar con pieza.actualizar_pieza_recortada(...),
    rotar con pieza.rotar(...), etc. Se llama una vez por pieza, antes de comparar."""
    ###COMPLETAR
    raise NotImplementedError

def comparar_GlobalNoise_ColorDegradation_Fourier_Geometry(pieza_a: Pieza, pieza_b: Pieza, relacion: str) -> float:
    """Costo de poner pieza_b junto a pieza_a (menor = mas compatibles). `relacion` es
    "horizontal" (B a la derecha de A) o "vertical" (B abajo de A); BORDES_ENFRENTADOS[relacion]
    da el par de lados que se tocan. Las piezas ya pasaron por preprocesar_GlobalNoise_ColorDegradation_Fourier_Geometry."""
    ###COMPLETAR
    raise NotImplementedError


In [ ]:
resultados_GlobalNoise_ColorDegradation_Fourier_Geometry = []
for i, ruta in enumerate(listar_imagenes("imagenes/seccion_9")):
    semilla = 900 + i

    rng = np.random.default_rng(seed=semilla)
    degradadores_color = [degradadorPorXoR, degradadorLuminancia, degradadorPorCrominancia]
    degradador = rng.choice(degradadores_color)

    degradador_seccion_9 = DegradadorPorPiezaCompuesto(degradador, degradador_seccion_3)

    rc = cargar_puzzle(ruta, semilla, tiene_ranuras=True, degradador_global=degradador_global_sin_impulsivo, degradador_por_pieza=degradador_seccion_9)
    grilla = resolver_seccion(rc, preprocesar_GlobalNoise_ColorDegradation_Fourier_Geometry, comparar_GlobalNoise_ColorDegradation_Fourier_Geometry)
    resultados_GlobalNoise_ColorDegradation_Fourier_Geometry.append({"imagen": ruta, "semilla": semilla, "vecindad": calcular_precision_vecindad(grilla, rc)})


In [ ]:
promedio_GlobalNoise_ColorDegradation_Fourier_Geometry = reportar_seccion("GlobalNoise_ColorDegradation_Fourier_Geometry", resultados_GlobalNoise_ColorDegradation_Fourier_Geometry)
assert promedio_GlobalNoise_ColorDegradation_Fourier_Geometry > 0.85, (
    f"Seccion GlobalNoise_ColorDegradation_Fourier_Geometry: el promedio de precision de vecindad ({promedio_GlobalNoise_ColorDegradation_Fourier_Geometry*100:.1f}%) "
    "debe superar el 85%."
)
print("Seccion GlobalNoise_ColorDegradation_Fourier_Geometry: OK")


## Sección 10 — GlobalNoise + ColorDegradation + Geometry + Rotation (sin sal y pimienta)

- Degradaciones, en este orden: `degradador_global_sin_impulsivo` (global) →
  `degradador_color_rotacion` (por pieza: color y después rotación con líneas al mismo ángulo).
- Piezas con encastres reales (`tiene_ranuras=True`), grilla 6x6.
- Implementar: `preprocesar_GlobalNoise_ColorDegradation_Geometry_Rotation(pieza)` y `comparar_GlobalNoise_ColorDegradation_Geometry_Rotation(pieza_a, pieza_b, relacion) -> float`.

In [ ]:
# Ejemplo de rompecabezas de esta seccion (primera imagen, semilla 1000)
rc_ejemplo_GlobalNoise_ColorDegradation_Geometry_Rotation = cargar_puzzle(listar_imagenes("imagenes/seccion_10")[0], 1000, tiene_ranuras=True, degradador_global=degradador_global_sin_impulsivo, degradador_por_pieza=degradador_color_rotacion)
mostrar_piezas_desordenadas_v2(rc_ejemplo_GlobalNoise_ColorDegradation_Geometry_Rotation, max_piezas=12, titulo="Sección 10 — GlobalNoise + ColorDegradation + Geometry + Rotation (sin sal y pimienta)")


In [ ]:
def preprocesar_GlobalNoise_ColorDegradation_Geometry_Rotation(pieza: Pieza) -> None:
    """Prepara `pieza` IN PLACE (no devuelve nada): limpiar con pieza.actualizar_pieza_recortada(...),
    rotar con pieza.rotar(...), etc. Se llama una vez por pieza, antes de comparar."""
    ###COMPLETAR
    raise NotImplementedError

def comparar_GlobalNoise_ColorDegradation_Geometry_Rotation(pieza_a: Pieza, pieza_b: Pieza, relacion: str) -> float:
    """Costo de poner pieza_b junto a pieza_a (menor = mas compatibles). `relacion` es
    "horizontal" (B a la derecha de A) o "vertical" (B abajo de A); BORDES_ENFRENTADOS[relacion]
    da el par de lados que se tocan. Las piezas ya pasaron por preprocesar_GlobalNoise_ColorDegradation_Geometry_Rotation."""
    ###COMPLETAR
    raise NotImplementedError


In [ ]:
resultados_GlobalNoise_ColorDegradation_Geometry_Rotation = []
for i, ruta in enumerate(listar_imagenes("imagenes/seccion_10")):
    semilla = 1000 + i

    rng = np.random.default_rng(seed=semilla)
    degradadores_color = [degradadorPorXoR, degradadorLuminancia, degradadorPorCrominancia]
    degradador = rng.choice(degradadores_color)

    degradador_seccion_10 = DegradadorPorPiezaCompuesto(degradador, degradador_seccion_5)

    rc = cargar_puzzle(ruta, semilla, tiene_ranuras=True, degradador_global=degradador_global_sin_impulsivo, degradador_por_pieza=degradador_seccion_10)
    grilla = resolver_seccion(rc, preprocesar_GlobalNoise_ColorDegradation_Geometry_Rotation, comparar_GlobalNoise_ColorDegradation_Geometry_Rotation)
    resultados_GlobalNoise_ColorDegradation_Geometry_Rotation.append({"imagen": ruta, "semilla": semilla, "vecindad": calcular_precision_vecindad(grilla, rc)})


In [ ]:
promedio_GlobalNoise_ColorDegradation_Geometry_Rotation = reportar_seccion("GlobalNoise_ColorDegradation_Geometry_Rotation", resultados_GlobalNoise_ColorDegradation_Geometry_Rotation)
assert promedio_GlobalNoise_ColorDegradation_Geometry_Rotation > 0.85, (
    f"Seccion GlobalNoise_ColorDegradation_Geometry_Rotation: el promedio de precision de vecindad ({promedio_GlobalNoise_ColorDegradation_Geometry_Rotation*100:.1f}%) "
    "debe superar el 85%."
)
print("Seccion GlobalNoise_ColorDegradation_Geometry_Rotation: OK")
